# 🎨 LittleNest — OpenRouter / Qwen Prompt Generator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janesh008/ETSY-pipeline/blob/main/prompt-generator/LittleNest_Prompt_Generator.ipynb)

> **Single-cell design** — paste this one cell into any existing Colab notebook.
> Edit the `CONFIG` block at the top, then run.

In [ ]:
# =============================================================================
#  LittleNest Prompt Generator — ALL-IN-ONE CELL
#  Edit the CONFIG block below, then run the cell.
# =============================================================================

# ─── CONFIG ──────────────────────────────────────────────────────────────────

CARTOON          = 'Jungle Book'     # Cartoon / franchise name
THEME            = 'birthday'        # Event theme  (birthday, baby shower …)
STYLE            = None              # Style hint — None = auto-infer
SECTIONS         = 'full bundle'     # 'full bundle' or comma-separated names
PROMPT_COUNT     = 130               # Total prompts to generate
REFERENCE_NOTES  = None              # Optional reference image notes

# Where to load the module from.
# 'drive'  → reads from DRIVE_MODULE_PATH below (mount Drive first).
# 'github' → shallow-clones the repo. Use just 'username/repo-name' format.
MODULE_SOURCE     = 'drive'
DRIVE_MODULE_PATH = '/content/drive/MyDrive/ETSY/ETSY-pipeline/prompt-generator'
GITHUB_REPO       = 'janesh008/ETSY-pipeline'   # username/repo only — no URL, no .git

# Where to save the output file.
OUTPUT_PATH = (
    '/content/drive/MyDrive/ETSY/output/'
    + CARTOON.replace(' ', '_').replace('&', 'and').lower()
    + f'_{THEME}_prompts.txt'
)

# ─── SETUP ───────────────────────────────────────────────────────────────────

import subprocess, sys, os

# Install dependencies for running local Qwen2.5-VL.
print('Installing dependencies (transformers, qwen-vl-utils, accelerate)...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'qwen-vl-utils', 'accelerate', 'torch', 'torchvision'], check=True)

# Load module from Drive or GitHub.
if MODULE_SOURCE == 'github':
    clone_target = '/content/etsy_pipeline'
    if not os.path.exists(clone_target):
        result = subprocess.run(
            ['git', 'clone', '--depth', '1',
             f'https://github.com/{GITHUB_REPO}.git', clone_target],
            capture_output=True, text=True
        )
        print(result.stdout or result.stderr)
    module_path = f'{clone_target}/prompt-generator'
else:
    module_path = DRIVE_MODULE_PATH

# Standardize path representation
module_path = os.path.abspath(module_path)

if module_path not in sys.path:
    sys.path.insert(0, module_path)

# Insert the parent directory too, just in case
parent_dir = os.path.dirname(module_path)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print(f'Module path : {module_path}')

# Pre-flight import check with a helpful error message
try:
    import importlib
    import qwen_client;             importlib.reload(qwen_client)
    import littlenest_prompt_generator; importlib.reload(littlenest_prompt_generator)
    import prompt_validator;         importlib.reload(prompt_validator)
except ImportError as e:
    raise ImportError(
        f'Failed to import pipeline modules: {e}\n'
        f'Ensure Google Drive is mounted and DRIVE_MODULE_PATH is correct, '
        f'or switch MODULE_SOURCE to "github".'
    )

# ─── INITIALIZE & TEST ────────────────────────────────────────────────────────

from qwen_client import quick_qwen_test

print('Initializing and testing local Qwen model (this may take a few minutes on first run)...')
test_resp = quick_qwen_test('Reply with OK only.')
print(f'Initialization & connection OK. Smoke test response: {test_resp.strip()}')

# ─── GENERATE ────────────────────────────────────────────────────────────────

from littlenest_prompt_generator import PromptRequest, generate_littlenest_prompts, save_prompt_batch

request = PromptRequest(
    cartoon         = CARTOON,
    theme           = THEME,
    style           = STYLE,
    sections        = SECTIONS,
    prompt_count    = PROMPT_COUNT,
    reference_notes = REFERENCE_NOTES,
)

print(f'\nGenerating {PROMPT_COUNT} prompts for "{CARTOON} — {THEME}" …')
prompt_text = generate_littlenest_prompts(request)
print(f'Done. Characters generated: {len(prompt_text):,}')

# ─── VALIDATE ────────────────────────────────────────────────────────────────

from prompt_validator import validate_prompt_batch, parse_prompt_batch

validation = validate_prompt_batch(prompt_text)
print(f'\nValidation passed : {validation.ok}')

if validation.issues:
    print('Issues:')
    for issue in validation.issues:
        print(f'  ⚠  {issue}')
else:
    print('No structural issues found.')

print('\nSection counts:')
for section, count in validation.section_counts.items():
    marker = '✅' if count >= 10 else ('➖' if count == 0 else '⚠️')
    print(f'  {marker}  {section:<30} {count}')

# Uncomment to hard-stop the cell on validation failure:
# validation.raise_if_invalid()

# ─── SAVE ────────────────────────────────────────────────────────────────────

saved = save_prompt_batch(prompt_text, OUTPUT_PATH)
print(f'\nSaved → {saved}')

# ─── PARSE (optional — use in downstream image loop) ─────────────────────────

prompts_by_section = parse_prompt_batch(prompt_text)
print(f'\nActive sections : {list(prompts_by_section.keys())}')

if 'MAIN_CHARACTER' in prompts_by_section:
    print('\nFirst MAIN_CHARACTER prompt preview:')
    print(prompts_by_section['MAIN_CHARACTER'][0])

# prompts_by_section is available for the rest of your notebook.
# Example: prompts_by_section['SCENE'][0]  → first scene prompt